# Notebook 13: Mechanistic QAT Pipeline

This notebook is where I actually apply my Grid-Aware Loss and unlearn each fact from my forget set, but targeted only at the specific MLP layer(s) that Notebook 10's causal tracing identified for that fact.

Since different facts get localized to different layers, I can't just do one giant training loop over the whole forget set. Instead, I group ("cluster") my forget set articles by their primary localized layer (the top-1 layer from `trace_map.json`), so I end up with one mini-dataset per layer, e.g. "all the facts whose knowledge lives mainly in Layer 6". Then I unfreeze only that one MLP layer and run my QAT loop just on that cluster, before moving to the next layer.

I set a `MIN_CLUSTER_SIZE` of 16, because some layers only have a handful of articles routed to them, and training on a batch of 2-3 examples for 5 epochs doesn't give a stable gradient signal - it risks overfitting or just being noisy. Any layer with fewer than 16 articles gets moved into a "residuals" bucket instead of being trained on its own.

**Input:** `trace_map.json` (from Notebook 10) and my fp16 target model.
**Output:** per-layer PyTorch `DataLoader`s, one per valid cluster, plus a residual dataloader for the leftover small clusters. The notebook then trains each cluster (and the residuals) and saves the final unlearned model to `QSurgical_Clustered_FP16`.

In [ ]:
import json
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# 1. Environment & Paths
MODEL_PATH = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"
INPUT_JSON_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/trace_map.json"
MIN_CLUSTER_SIZE = 16  # Minimum batching threshold to avoid training on too few examples for a stable gradient signal.
BATCH_SIZE = 4         # Safe batch size for Colab A100 environment

# 2. PyTorch Dataset Class
class MUSE_Dataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.texts = texts
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encodings = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze()
        }

# 3. Load Tokenizer
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4. Load and Route the JSON Data
print("\n--- Initializing Data Router ---")
with open(INPUT_JSON_PATH, "r") as f:
    trace_map = json.load(f)

# Initialize raw cluster bins (32 layers)
raw_clusters = {layer: [] for layer in range(32)}

for article_id, data in trace_map.items():
    # Strategy 3 groups by the PRIMARY (index 0) layer responsible for the fact
    primary_layer = data['top_layers'][0]
    raw_clusters[primary_layer].append(data['text'])

# 5. Filter and Build DataLoaders
cluster_dataloaders = {}
residual_texts = []

print("\n📊 Cluster Distribution Analysis:")
print("-" * 40)
print(f"{'Layer ID':<10} | {'Total Articles':<15} | {'Status'}")
print("-" * 40)

for layer, texts in raw_clusters.items():
    count = len(texts)

    if count == 0:
        continue # Skip empty layers entirely

    if count >= MIN_CLUSTER_SIZE:
        print(f"Layer {layer:<5} | {count:<15} | ✅ Validated for Batched QAT")
        dataset = MUSE_Dataset(texts, tokenizer)
        # We shuffle to ensure batches within the cluster are randomized
        cluster_dataloaders[layer] = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    else:
        print(f"Layer {layer:<5} | {count:<15} | ⚠️ Too small. Moved to Residuals.")
        residual_texts.extend(texts)

# 6. Handle the Residual Dataset
if len(residual_texts) > 0:
    print("-" * 40)
    print(f"Residuals  | {len(residual_texts):<15} | 🔄 Grouped into catch-all loader")
    residual_dataset = MUSE_Dataset(residual_texts, tokenizer)
    residual_dataloader = DataLoader(residual_dataset, batch_size=BATCH_SIZE, shuffle=True)
else:
    residual_dataloader = None

print("\n✅ Data Routing Complete!")
print(f"Total Valid Layer Clusters: {len(cluster_dataloaders)}")

Loading Tokenizer...

--- Initializing Data Router ---

📊 Cluster Distribution Analysis:
----------------------------------------
Layer ID   | Total Articles  | Status
----------------------------------------
Layer 0     | 117             | ✅ Validated for Batched QAT
Layer 1     | 76              | ✅ Validated for Batched QAT
Layer 2     | 160             | ✅ Validated for Batched QAT
Layer 3     | 18              | ✅ Validated for Batched QAT
Layer 4     | 22              | ✅ Validated for Batched QAT
Layer 5     | 21              | ✅ Validated for Batched QAT
Layer 6     | 65              | ✅ Validated for Batched QAT
Layer 7     | 39              | ✅ Validated for Batched QAT
Layer 8     | 22              | ✅ Validated for Batched QAT
Layer 9     | 14              | ⚠️ Too small. Moved to Residuals.
Layer 10    | 16              | ✅ Validated for Batched QAT
Layer 11    | 11              | ⚠️ Too small. Moved to Residuals.
Layer 12    | 5               | ⚠️ Too small. Moved to Resi

## Grid-Aware Loss and Layer-by-Layer QAT Training

This is my implementation of the custom loss function from my methodology, combined with the actual training loop.

My loss function has three parts:
1. **Forget loss** - standard Gradient Ascent: I take the cross-entropy loss on the forget batch and negate it (so the optimizer is pushed to *increase* the loss on the forgotten fact instead of decreasing it, i.e. make the model worse at predicting it). I clamp this at -50 so it doesn't blow up to a huge negative number and destabilize training.
2. **Retain loss** - normal language modeling cross-entropy loss on a batch from my retain set, weighted by `alpha_retain`, so the model doesn't just forget everything and start producing garbage while I'm erasing one specific fact.
3. **Grid penalty (my main contribution)** - this forces the weight change `Δw` between the original and current weights to be at least as large as the quantization bucket size (`dynamic_grid_margin`), so that after 4-bit quantization the update survives instead of being rounded away ("bucket collapse"). `dynamic_grid_margin` is now calculated per layer from the layer's own weight range (`(2 * layer_max_abs) / 15`), rather than a fixed constant, so each MLP layer gets a margin that actually reflects its own 4-bit quantization step size.

**Fix I made to the penalty calculation:** originally I applied the grid penalty as a mean over every weight in the whole MLP block (potentially millions of parameters). The problem was that only a small number of weights actually need to move to unlearn one fact, so the mean was almost entirely dominated by the untouched weights and barely moved during training, regardless of whether the important weights were shifting or not. I fixed this by only applying the penalty to the top-5000 weights with the largest change (`top_k_neurons=5000`), so the loss signal is now concentrated on the weights that are actually doing the work. I also changed my checkpoint selection to track `max_shift` directly (the largest single weight movement seen so far) instead of the old diluted mean penalty, since that's a much more meaningful proxy for "did I actually get closer to a real quantization jump."

I process layers **top-down** (Layer 31 first, down to Layer 0), unfreezing only the target MLP layer's parameters each time, training for 10 epochs on that layer's cluster (bumped up from 5 after seeing weight movement was too slow), and keeping the checkpoint with the largest max shift seen during training before moving to the next layer.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
import gc
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Define Utility-Preserving Unlearning Loss
class UtilityPreservingUnlearningLoss(nn.Module):
    def __init__(self, lambda_reg=100.0, alpha_retain=20.0, top_k_neurons=5000):
        super().__init__()
        self.lambda_reg = lambda_reg
        self.alpha_retain = alpha_retain
        self.top_k_neurons = top_k_neurons

    def forward(self, forget_logits, forget_labels, retain_logits, retain_labels, current_weights, original_weights, dynamic_margin):
        # 1. Forget Loss
        shift_f_logits = forget_logits[..., :-1, :].contiguous().float()
        shift_f_labels = forget_labels[..., 1:].contiguous()
        f_ce_loss = F.cross_entropy(shift_f_logits.view(-1, shift_f_logits.size(-1)), shift_f_labels.view(-1))
        forget_loss = -torch.clamp(f_ce_loss, max=50.0)

        # 2. Retain Loss
        shift_r_logits = retain_logits[..., :-1, :].contiguous().float()
        shift_r_labels = retain_labels[..., 1:].contiguous()
        retain_loss = F.cross_entropy(shift_r_logits.view(-1, shift_r_logits.size(-1)), shift_r_labels.view(-1), ignore_index=tokenizer.pad_token_id)

        # 3. Sparse Top-K Grid Penalty
        weight_diff = torch.abs(current_weights - original_weights).view(-1)

        # Dynamically isolate the exact neurons trying to unlearn the fact
        k = min(self.top_k_neurons, weight_diff.size(0))
        top_k_diff, _ = torch.topk(weight_diff, k)

        # Apply the margin penalty strictly to these specific neurons
        grid_penalty = torch.relu(dynamic_margin - top_k_diff).mean()

        total_loss = forget_loss + (self.alpha_retain * retain_loss) + (self.lambda_reg * grid_penalty)
        return total_loss, forget_loss, retain_loss, grid_penalty, weight_diff.max()

# 2. Setup Training Environment
print("\nLoading Retain DataLoader for Utility Anchoring...")
RETAIN_SET_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/retain_set1.csv"
retain_df = pd.read_csv(RETAIN_SET_PATH)
retain_texts = retain_df['text'].fillna("").astype(str).tolist()
retain_dataset = MUSE_Dataset(retain_texts, tokenizer)
retain_dataloader = DataLoader(retain_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("\nLoading Base Phi-3 Model to GPU...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    dtype=torch.float16,
    device_map={"": 0}
)

criterion = UtilityPreservingUnlearningLoss()
scaler = torch.amp.GradScaler('cuda')

def get_flat_weights(modules):
    return torch.cat([p.view(-1) for m in modules for p in m.parameters()])

# 3. Modular Training Function
def train_layer_cluster(model, target_layer, dataloader, retain_dataloader, criterion, scaler, epochs=10): # Bumped to 10 epochs
    num_batches = len(dataloader)
    retain_iter = iter(retain_dataloader)

    # Freeze entire model first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze specific MLP block
    mlp_module = model.model.layers[target_layer].mlp
    mlp_module.to(torch.float32)
    for param in mlp_module.parameters():
        param.requires_grad = True

    target_modules = [mlp_module]
    original_block_weights = get_flat_weights(target_modules).clone().detach()

    # Dynamic Margin Calculation
    layer_max_abs = original_block_weights.abs().max().item()
    dynamic_grid_margin = (2 * layer_max_abs) / 15.0
    print(f"  [Layer {target_layer} | Max W: {layer_max_abs:.4f} | Margin Target: {dynamic_grid_margin:.6f}]")

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
    model.train()

    best_top_k_shift = 0.0 # FIX: Selecting checkpoint based on actual movement, not the diluted penalty
    best_weights = None

    for epoch in range(epochs):
        epoch_loss, epoch_f_loss, epoch_r_loss, epoch_g_penalty, epoch_max_shift = 0.0, 0.0, 0.0, 0.0, 0.0

        for step, forget_batch in enumerate(dataloader):
            optimizer.zero_grad()
            f_inputs, f_attn = forget_batch['input_ids'].to("cuda"), forget_batch['attention_mask'].to("cuda")

            try:
                retain_batch = next(retain_iter)
            except StopIteration:
                retain_iter = iter(retain_dataloader)
                retain_batch = next(retain_iter)

            r_inputs, r_attn = retain_batch['input_ids'].to("cuda"), retain_batch['attention_mask'].to("cuda")

            with torch.autocast("cuda", dtype=torch.float16):
                f_outputs = model(input_ids=f_inputs, attention_mask=f_attn)
                r_outputs = model(input_ids=r_inputs, attention_mask=r_attn)
                current_block_weights = get_flat_weights(target_modules)

                loss, f_loss, r_loss, g_penalty, max_shift = criterion(
                    f_outputs.logits, f_inputs, r_outputs.logits, r_inputs,
                    current_block_weights, original_block_weights, dynamic_grid_margin
                )

            if math.isnan(loss.item()):
                print(f"⚠️ NaN at Epoch {epoch+1}. Stopping layer early.")
                break

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            epoch_f_loss += f_loss.item()
            epoch_r_loss += r_loss.item()
            epoch_g_penalty += g_penalty.item()
            epoch_max_shift = max(epoch_max_shift, max_shift.item())

            # Save checkpoint if weights achieved a higher maximum shift
            if max_shift.item() > best_top_k_shift:
                best_top_k_shift = max_shift.item()
                best_weights = [copy.deepcopy(m.state_dict()) for m in target_modules]

        pct_reached = (epoch_max_shift / dynamic_grid_margin) * 100 if dynamic_grid_margin > 0 else 0
        print(f"  -> Ep {epoch+1}/{epochs} | F-Loss: {epoch_f_loss/num_batches:.1f} | "
              f"R-Loss: {epoch_r_loss/num_batches:.1f} | Max Shift: {epoch_max_shift:.5f} ({pct_reached:.1f}% of Target)")

    # Restore best quantized state
    if best_weights is not None:
        for idx, m in enumerate(target_modules):
            m.load_state_dict(best_weights[idx])

    mlp_module.to(torch.float16)
    torch.cuda.empty_cache()
    gc.collect()

# 4. Execution Flow
print("\n--- Starting Top-Down Clustered Unlearning ---")
sorted_layers = sorted(cluster_dataloaders.keys(), reverse=True)

# Train Validated Clusters
for target_layer in sorted_layers:
    print(f"\n[+] Processing Layer {target_layer} Cluster | {len(cluster_dataloaders[target_layer])} batches")
    train_layer_cluster(model, target_layer, cluster_dataloaders[target_layer], retain_dataloader, criterion, scaler, epochs=10)

# FIX: Train Residual Clusters for real
if residual_dataloader is not None:
    fallback_layer = 31 # Standard deep-semantic fallback layer
    print(f"\n[+] Processing Residuals | {len(residual_dataloader)} batches on default Layer {fallback_layer}")
    train_layer_cluster(model, fallback_layer, residual_dataloader, retain_dataloader, criterion, scaler, epochs=10)

# 5. Save the Final Model
# OUTPUT_MODEL_DIR = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/QSurgical_Clustered_FP16"
OUTPUT_MODEL_DIR = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/QAMU1_FP16"
model.save_pretrained(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"\n✅ Full Layer-Clustered Q-Surgical Framework Complete! Model saved to {OUTPUT_MODEL_DIR}")


Loading Retain DataLoader for Utility Anchoring...

Loading Base Phi-3 Model to GPU...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]


--- Starting Top-Down Clustered Unlearning ---

[+] Processing Layer 31 Cluster | 22 batches
  [Layer 31 | Max W: 2.2656 | Margin Target: 0.302083]
  -> Ep 1/10 | F-Loss: -2.9 | R-Loss: 2.2 | Max Shift: 0.00484 (1.6% of Target)
  -> Ep 2/10 | F-Loss: -4.1 | R-Loss: 2.1 | Max Shift: 0.01231 (4.1% of Target)
  -> Ep 3/10 | F-Loss: -7.9 | R-Loss: 2.1 | Max Shift: 0.02034 (6.7% of Target)
  -> Ep 4/10 | F-Loss: -11.0 | R-Loss: 2.1 | Max Shift: 0.02860 (9.5% of Target)
  -> Ep 5/10 | F-Loss: -12.0 | R-Loss: 2.1 | Max Shift: 0.03680 (12.2% of Target)
  -> Ep 6/10 | F-Loss: -13.0 | R-Loss: 2.2 | Max Shift: 0.04620 (15.3% of Target)
  -> Ep 7/10 | F-Loss: -11.5 | R-Loss: 2.1 | Max Shift: 0.05529 (18.3% of Target)
  -> Ep 8/10 | F-Loss: -12.2 | R-Loss: 2.2 | Max Shift: 0.06432 (21.3% of Target)
  -> Ep 9/10 | F-Loss: -11.9 | R-Loss: 2.0 | Max Shift: 0.07307 (24.2% of Target)
  -> Ep 10/10 | F-Loss: -12.3 | R-Loss: 2.1 | Max Shift: 0.08173 (27.1% of Target)

[+] Processing Layer 30 Cluster | 7 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Full Layer-Clustered Q-Surgical Framework Complete! Model saved to /content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/QAMU1_FP16
